# Medallion demo, part 1: generate the Bronze test data

This notebook writes **raw source extracts** of an online retailer ("Oog Shop") as Parquet files to the S3 storage (RustFS). They are the
**Bronze layer files** that `02-medallion-pipeline.ipynb` ingests, cleans (Silver) and models as a star schema (Gold), always inside DuckLake.

* **Domain:** E-commerce / retail, 10 source tables, one example month (September 2026, one extract per day).
* **Volumes** (`SCALE = 1.0`): the three biggest daily tables carry **10,000 returns, 100,000 orders and 1,000,000 order items per day**
  (plus 100,000 payments); the master data tables have 10,000 to 300,000 rows.
* **Layout:** `s3://oog-lake/bronze/<table>/load_date=YYYY-MM-DD/<table>.parquet`. The folder name is the load date, so incremental loads can
  be simulated by pointing the pipeline at another date.
* **Deterministic:** all values come from hash functions of the row ids, never from `random()`. Running this notebook again writes
  exactly the same content (existing files are overwritten).
* **Realistic and dirty:** the extracts contain what real source systems deliver - mixed spellings and date formats, NULLs, duplicates,
  invalid values, orphan references, and later corrections (updates and deletes of rows delivered on earlier days). One source gets a new
  column in the middle of the month (schema evolution).

| Table | Kind | Delivery | Rows (per file) |
|---|---|---|---|
| `customers` | master, change feed | initial load on day 1, then new + changed rows daily | 200,000 initial, about 5,000 per day |
| `addresses` | master, change feed | like `customers` | 300,000 initial, about 2,000 per day |
| `products` | master, change feed | like `customers` | 50,000 initial, about 550 per day |
| `suppliers` | master, snapshot | full snapshot every 7 days (days 1, 8, 15, 22, 29) | 10,000 |
| `stores` | master, snapshot | full snapshot every 7 days | 10,000 |
| `promotions` | master, snapshot | full snapshot every 7 days | 10,000 |
| `orders` | transactions | every day: new orders + status changes + deletions | about 103,000 |
| `order_items` | transactions | every day: new items + corrections + deletions | about 1,003,000 |
| `payments` | transactions | every day | about 99,000 |
| `returns` | transactions | every day | about 10,500 |

Every change feed row has `op` (`I` insert, `U` update, `D` delete) and `updated_at`.

In [ ]:
from datetime import date, timedelta

SCALE = 1.0                            # 1.0 = full volumes; e.g. 0.01 for a quick test run
MONTH_START = date(2026, 9, 1)         # day 1 of the example month
DAYS = 30
BRONZE = "s3://oog-lake/bronze"        # root of the Bronze Parquet files
SCHEMA_CHANGE_DATE = date(2026, 9, 15) # from this day on the customers extract has an extra column: loyalty_tier


def scaled(base, minimum=1):
    return max(minimum, int(round(base * SCALE)))


N = {
    "customers": scaled(200_000), "customers_new": scaled(1_000), "customers_upd": scaled(4_000),
    "addr_upd": scaled(1_500),
    "products": scaled(50_000), "products_new": scaled(50), "products_upd": scaled(500),
    "suppliers": scaled(10_000), "suppliers_new": scaled(20),
    "stores": scaled(10_000), "promotions": scaled(10_000), "promotions_new": scaled(200),
    "orders": scaled(100_000), "items": scaled(1_000_000), "returns": scaled(10_000),
}
# one extract per day; the day number d runs from 1 to DAYS
def day_date(d):
    return MONTH_START + timedelta(days=d - 1)


def is_snapshot_day(d):
    return (d - 1) % 7 == 0


print({k: f"{v:,}" for k, v in N.items()})
print("days:", day_date(1), "to", day_date(DAYS), "| writing to", BRONZE)

## DuckDB connection and helper macros

The data is generated with DuckDB SQL (no DuckLake involved yet). The macros `rnd` (a repeatable pseudo random number between 0 and 1, derived
from an id and a salt), `pick` (choose a list element), `secs` (a pseudo random time of day) and the two price functions make the data
deterministic. `oog.duck.connect(attach=False)` opens DuckDB with the S3 secret for RustFS but without the DuckLake catalog.

In [ ]:
from oog import duck

con = duck.connect(attach=False)


def set_memory_limit(connection, fraction=0.6):
    """DuckDB sizes its memory by the machine, not by the container limit. Without a limit the container's OOM killer ends the kernel on big days;
    with it DuckDB spills to disk instead."""
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            value = open(path).read().strip()
        except OSError:
            continue
        if value.isdigit() and int(value) < 1 << 40:
            limit_mb = int(int(value) / 1024 / 1024 * fraction)
            connection.execute(f"SET memory_limit = '{limit_mb}MB'").fetchall()
            print(f"DuckDB memory limit: {limit_mb:,} MB (container limit {int(value) / 1024 ** 3:.0f} GiB)")
            return


set_memory_limit(con)

con.execute("CREATE OR REPLACE MACRO rnd(x, s) AS (hash(CAST(x AS BIGINT) * 1000003 + s) % 1000000) / 1000000.0")
con.execute("CREATE OR REPLACE MACRO pick(arr, x, s) AS arr[1 + CAST(floor(rnd(x, s) * len(arr)) AS BIGINT)]")
con.execute("CREATE OR REPLACE MACRO secs(x, s) AS to_seconds(CAST(floor(rnd(x, s) * 86400) AS BIGINT))")
con.execute("CREATE OR REPLACE MACRO base_price(pid) AS round(1 + 40 * power(rnd(pid, 3), 5), 2)")


def L(items):
    """Python list -> SQL list literal."""
    def lit(x):
        if x is None:
            return "NULL"
        if isinstance(x, str):
            return "'" + x.replace("'", "''") + "'"
        return str(x)
    return "[" + ", ".join(lit(x) for x in items) + "]"


print("DuckDB", con.execute("SELECT version()").fetchall()[0][0], "| macros ready")

## Reference lists

Value pools for names, places, categories and so on. Some entries are deliberately "dirty" (different spellings of the same thing);
the Silver layer has to clean them up.

In [ ]:
FIRST = ["Anna", "Lukas", "Marie", "Paul", "Sophie", "Jonas", "Laura", "Felix", "Emma", "Leon", "Mia", "Maximilian", "Hannah", "Elias",
         "Lea", "Noah", "Lena", "Finn", "Julia", "Tim", "Clara", "Ben", "Johanna", "Moritz", "Nina", "David", "Sarah", "Jan", "Katharina", "Tom"]
LAST = ["Mueller", "Schmidt", "Schneider", "Fischer", "Weber", "Meyer", "Wagner", "Becker", "Schulz", "Hoffmann", "Koch", "Bauer", "Richter",
        "Klein", "Wolf", "Schroeder", "Neumann", "Schwarz", "Zimmermann", "Braun", "Krueger", "Hartmann", "Lange", "Schmitt", "Werner",
        "Krause", "Meier", "Lehmann", "Huber", "Fuchs"]
DOMAINS = ["gmail.com", "web.de", "gmx.de", "outlook.com", "t-online.de", "yahoo.com", "icloud.com", "example.org"]
AREAS = ["170", "171", "151", "160", "175", "30", "40", "89"]
COUNTRY_RAW = ["DE"] * 10 + ["AT", "AT", "CH", "CH", "NL", "FR", "Deutschland", "germany", "Österreich", "Switzerland", "de", "DE "]
ADDR_COUNTRY_RAW = ["DE"] * 12 + ["AT", "CH", "NL", "FR", "Deutschland"]
GENDER_RAW = ["M", "F", "m", "f", "male", "female", "Male", "Female", "d", "D", None]
SEGMENT_RAW = ["consumer"] * 4 + ["premium", "vip", "b2b", "Consumer", "PREMIUM"]
STREETS = ["Hauptstrasse", "Bahnhofstrasse", "Gartenweg", "Schulstrasse", "Dorfstrasse", "Birkenweg", "Waldstrasse", "Lindenallee",
           "Ringstrasse", "Kirchplatz", "Marktplatz", "Berliner Strasse", "Muenchner Strasse", "Hamburger Allee"]
CITIES = ["Berlin", "Hamburg", "Muenchen", "Koeln", "Frankfurt", "Stuttgart", "Duesseldorf", "Leipzig", "Dortmund", "Essen", "Bremen",
          "Dresden", "Hannover", "Nuernberg", "Wien", "Zuerich", "Graz", "Basel"]
CATEGORIES = ["Electronics", "Home & Garden", "Fashion", "Sports", "Toys", "Books", "Beauty", "Grocery"]
SUBCATS = ["Accessories", "Premium", "Basic", "Outdoor", "Kids", "Office"]
BRANDS = ["Nordwind", "Alpina", "Kronos", "Lumina", "Veltro", "Brenner", "Aurum", "Sanora", "Tiefblau", "Marlo", "Pixelfox", "Greenleaf"]
ADJ = ["Smart", "Classic", "Compact", "Ultra", "Eco", "Sport", "Urban", "Deluxe", "Basic", "Pro"]
NOUNS = ["Lamp", "Jacket", "Backpack", "Headphones", "Blender", "Notebook", "Sneakers", "Watch", "Kettle", "Speaker", "Mug", "Desk", "Tent", "Camera"]
SUP_A = ["Nord", "Sued", "Ost", "West", "Euro", "Inter", "Global", "Prima", "Alpha", "Delta"]
SUP_B = ["Handel", "Logistik", "Trading", "Import", "Vertrieb", "Werke", "Textil", "Elektro"]
LEGAL = ["GmbH", "AG", "Ltd", "BV", "SARL", "KG"]
CHANNELS = ["web", "web", "web", "app", "app", "store", "store", "phone"]
PAY_RAW = ["credit_card", "CREDIT CARD", "Visa", "mastercard", "paypal", "PayPal", "sepa", "SEPA Lastschrift", "klarna", "invoice", "apple pay"]
RETURN_REASONS = ["damaged", "wrong_item", "not_as_described", "changed_mind", "too_late", None]
STORE_TYPES = ["flagship", "outlet", "pickup", "outlet", "pickup", "Flagship"]
print("lists ready")

## Writing the files

`write_day` writes the result of one query as a single Parquet file (zstd) to the day's folder and counts the rows afterwards by reading the
file's metadata. Re-running overwrites the file.

In [ ]:
import time

written = []   # (table, load_date, rows)


def write_day(table, d, query):
    ld = day_date(d)
    path = f"{BRONZE}/{table}/load_date={ld}/{table}.parquet"
    t0 = time.time()
    con.execute(f"COPY ({query}) TO '{path}' (FORMAT parquet, COMPRESSION zstd)")
    rows = con.execute(f"SELECT num_rows FROM parquet_file_metadata('{path}')").fetchall()[0][0]
    written.append((table, ld, rows))
    print(f"{table:12} {ld}  {rows:>9,} rows  {time.time() - t0:5.1f}s")
    return rows

## Table 1: `customers` (change feed)

Day 1 is the initial load of all customers. Every following day brings new customers (`I`) and changed customers (`U`: phone, segment,
loyalty tier). Real source systems also send duplicates, so a small share of the update rows is delivered twice.
Dirty on purpose: upper/lower case and blanks in names and e-mail addresses, e-mail addresses without `@`, three phone formats, three birth
date formats plus `0000-00-00`, several spellings for gender and country, mixed case segments.
From `SCHEMA_CHANGE_DATE` on the extract has the extra column `loyalty_tier`.

In [ ]:
def cust_before(d):
    """Number of customers that exist when day d starts."""
    return N["customers"] + max(0, d - 2) * N["customers_new"]


def customers_query(d):
    C0, CN, CU = N["customers"], N["customers_new"], N["customers_upd"]
    if d == 1:
        ids = f"SELECT i AS rn, i AS cid, 0 AS ver, 'I' AS op FROM range(1, {C0} + 1) t(i)"
    else:
        new = f"SELECT i AS rn, {C0} + ({d} - 2) * {CN} + i AS cid, 0 AS ver, 'I' AS op FROM range(1, {CN} + 1) t(i)"
        upd = (f"SELECT i AS rn, 1 + CAST(floor(rnd(i, 7000 + {d}) * {cust_before(d)}) AS BIGINT) AS cid, {d} AS ver, 'U' AS op "
               f"FROM range(1, {CU} + 1) t(i)")
        dup = f"SELECT rn, cid, ver, op FROM ({upd}) WHERE rn <= {max(1, int(CU * 0.003))}"
        ids = f"{new} UNION ALL {upd} UNION ALL {dup}"
    ld = day_date(d)
    tier = f", pick({L(['bronze', 'silver', 'gold', 'platinum', None])}, cid, 50 + ver) AS loyalty_tier" if ld >= SCHEMA_CHANGE_DATE else ""
    return f"""
    WITH ids AS ({ids}),
    b AS (
        SELECT rn, cid, ver, op,
               pick({L(FIRST)}, cid, 1) AS fn, pick({L(LAST)}, cid, 2) AS ln,
               pick({L(AREAS)}, cid, 4) AS area, CAST(1000000 + hash(cid * 31 + ver) % 8999999 AS VARCHAR) AS num,
               DATE '1950-01-01' + CAST(floor(rnd(cid, 6) * 20000) AS INTEGER) AS dob,
               pick({L(DOMAINS)}, cid, 3) AS dom
        FROM ids)
    SELECT
        cid AS customer_id,
        CASE WHEN rnd(cid, 31) < 0.03 THEN upper(fn) WHEN rnd(cid, 31) < 0.05 THEN ' ' || fn || ' ' ELSE fn END AS first_name,
        CASE WHEN rnd(cid, 32) < 0.03 THEN lower(ln) ELSE ln END AS last_name,
        CASE WHEN rnd(cid, 33) < 0.02 THEN NULL
             WHEN rnd(cid, 33) < 0.04 THEN upper(lower(fn) || '.' || lower(ln) || cid || '@' || dom)
             WHEN rnd(cid, 33) < 0.06 THEN ' ' || lower(fn) || '.' || lower(ln) || cid || '@' || dom || ' '
             WHEN rnd(cid, 33) < 0.07 THEN lower(fn) || '.' || lower(ln) || cid || dom
             ELSE lower(fn) || '.' || lower(ln) || cid || '@' || dom END AS email,
        CASE WHEN rnd(cid, 34) < 0.10 THEN NULL
             ELSE CASE CAST(floor(rnd(cid + ver, 35) * 3) AS INTEGER)
                    WHEN 0 THEN '+49 ' || area || ' ' || num
                    WHEN 1 THEN '0' || area || '-' || num
                    ELSE '(0' || area || ') ' || num END END AS phone,
        CASE WHEN rnd(cid, 36) < 0.03 THEN NULL
             WHEN rnd(cid, 36) < 0.035 THEN '0000-00-00'
             WHEN rnd(cid, 36) < 0.60 THEN strftime(dob, '%Y-%m-%d')
             WHEN rnd(cid, 36) < 0.90 THEN strftime(dob, '%d.%m.%Y')
             ELSE strftime(dob, '%m/%d/%Y') END AS birth_date,
        pick({L(GENDER_RAW)}, cid, 7) AS gender,
        pick({L(COUNTRY_RAW)}, cid, 8) AS country,
        pick({L(SEGMENT_RAW)}, cid, 10 + ver) AS segment,
        CASE WHEN cid <= {N['customers']} THEN TIMESTAMP '2021-09-01' + to_seconds(CAST(floor(rnd(cid, 8) * 157680000) AS BIGINT))
             ELSE CAST(DATE '{MONTH_START}' + CAST((cid - {N['customers']} - 1) // {N['customers_new']} + 1 AS INTEGER) AS TIMESTAMP) + secs(cid, 8)
        END AS registered_at,
        CASE WHEN op = 'I' THEN
                  CASE WHEN cid <= {N['customers']} THEN TIMESTAMP '2021-09-01' + to_seconds(CAST(floor(rnd(cid, 8) * 157680000) AS BIGINT))
                       ELSE CAST(DATE '{MONTH_START}' + CAST((cid - {N['customers']} - 1) // {N['customers_new']} + 1 AS INTEGER) AS TIMESTAMP) + secs(cid, 8) END
             ELSE TIMESTAMP '{ld}' + secs(rn, 9 + {d}) END AS updated_at,
        op{tier}
    FROM b
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("customers", d, customers_query(d))

## Table 2: `addresses` (change feed)

Every customer has a billing address; about half of the customers also have a shipping address. New customers bring new addresses,
and a few existing addresses change every day (street, house number). Dirty: postal codes without the leading zero, with blanks or missing,
upper case city names, different country spellings.

In [ ]:
def addresses_query(d):
    C0, CN, AU = N["customers"], N["customers_new"], N["addr_upd"]
    if d == 1:
        base = f"SELECT i AS rn, i AS cid, 0 AS ver, 'I' AS op FROM range(1, {C0} + 1) t(i)"
        ids = (f"SELECT rn, cid, 0 AS kind, ver, op FROM ({base}) "
               f"UNION ALL SELECT rn, cid, 1 AS kind, ver, op FROM ({base}) WHERE rnd(cid, 60) < 0.5")
    else:
        base = f"SELECT i AS rn, {C0} + ({d} - 2) * {CN} + i AS cid, 0 AS ver, 'I' AS op FROM range(1, {CN} + 1) t(i)"
        upd = (f"SELECT i AS rn, 1 + CAST(floor(rnd(i, 7500 + {d}) * {cust_before(d)}) AS BIGINT) AS cid, {d} AS ver, 'U' AS op "
               f"FROM range(1, {AU} + 1) t(i)")
        ids = (f"SELECT rn, cid, 0 AS kind, ver, op FROM ({base}) "
               f"UNION ALL SELECT rn, cid, 1 AS kind, ver, op FROM ({base}) WHERE rnd(cid, 60) < 0.5 "
               f"UNION ALL SELECT rn, cid, CASE WHEN rnd(cid, 60) < 0.5 AND rnd(rn, 61 + {d}) < 0.5 THEN 1 ELSE 0 END AS kind, ver, op FROM ({upd})")
    ld = day_date(d)
    return f"""
    WITH ids AS ({ids}),
    b AS (SELECT *, cid * 2 + kind AS aid, CASE kind WHEN 0 THEN cid ELSE 5000000 + cid END AS address_id FROM ids),
    c AS (SELECT *, 1000 + CAST(hash(cid * 7 + kind) % 98000 AS BIGINT) AS pc FROM b)
    SELECT
        address_id,
        cid AS customer_id,
        CASE WHEN kind = 0 THEN CASE WHEN rnd(aid, 62) < 0.03 THEN 'BILLING' ELSE 'billing' END
             ELSE CASE WHEN rnd(aid, 62) < 0.03 THEN 'Shipping' ELSE 'shipping' END END AS address_type,
        pick({L(STREETS)}, aid, 70 + ver) AS street,
        CAST(1 + floor(rnd(aid, 71 + ver) * 150) AS INTEGER) || CASE WHEN rnd(aid, 72) < 0.1 THEN 'a' ELSE '' END AS house_no,
        CASE WHEN rnd(aid, 74) < 0.01 THEN NULL
             WHEN rnd(aid, 74) < 0.03 THEN CAST(pc AS VARCHAR)
             WHEN rnd(aid, 74) < 0.05 THEN substr(lpad(CAST(pc AS VARCHAR), 5, '0'), 1, 2) || ' ' || substr(lpad(CAST(pc AS VARCHAR), 5, '0'), 3)
             ELSE lpad(CAST(pc AS VARCHAR), 5, '0') END AS postal_code,
        CASE WHEN rnd(aid, 75) < 0.03 THEN upper(pick({L(CITIES)}, cid, 76)) ELSE pick({L(CITIES)}, cid, 76) END AS city,
        pick({L(ADDR_COUNTRY_RAW)}, cid, 73) AS country,
        kind = 0 AS is_primary,
        CASE WHEN op = 'I' THEN
                  CASE WHEN cid <= {N['customers']} THEN TIMESTAMP '2021-09-01' + to_seconds(CAST(floor(rnd(cid, 8) * 157680000) AS BIGINT))
                       ELSE CAST(DATE '{MONTH_START}' + CAST((cid - {N['customers']} - 1) // {N['customers_new']} + 1 AS INTEGER) AS TIMESTAMP) + secs(cid, 8) END
             ELSE TIMESTAMP '{ld}' + secs(rn, 77 + {d}) END AS updated_at,
        op
    FROM c
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("addresses", d, addresses_query(d))

## Table 3: `products` (change feed)

Initial catalog on day 1, then new products and price changes (`U`, plus or minus up to 10 %) every day. Dirty: category spelled in
different cases, negative prices (a data entry bug), missing EAN codes.

In [ ]:
def prod_at(d):
    """Number of products after the deliveries of day d."""
    return N["products"] + max(0, d - 1) * N["products_new"]


def products_query(d):
    P0, PN, PU = N["products"], N["products_new"], N["products_upd"]
    if d == 1:
        ids = f"SELECT i AS rn, i AS pid, 0 AS ver, 'I' AS op FROM range(1, {P0} + 1) t(i)"
    else:
        new = f"SELECT i AS rn, {P0} + ({d} - 2) * {PN} + i AS pid, 0 AS ver, 'I' AS op FROM range(1, {PN} + 1) t(i)"
        upd = (f"SELECT i AS rn, 1 + CAST(floor(rnd(i, 7900 + {d}) * {prod_at(d - 1)}) AS BIGINT) AS pid, {d} AS ver, 'U' AS op "
               f"FROM range(1, {PU} + 1) t(i)")
        ids = f"{new} UNION ALL {upd}"
    ld = day_date(d)
    return f"""
    WITH ids AS ({ids}),
    b AS (SELECT *, pick({L(CATEGORIES)}, pid, 4) AS cat,
                 round(base_price(pid) * CASE WHEN ver = 0 THEN 1.0 ELSE 1 + (CAST(hash(pid * 13 + ver) % 21 AS INTEGER) - 10) / 100.0 END, 2) AS price
          FROM ids)
    SELECT
        pid AS product_id,
        'SKU-' || lpad(CAST(pid AS VARCHAR), 7, '0') AS sku,
        pick({L(ADJ)}, pid, 1) || ' ' || pick({L(NOUNS)}, pid, 2) || ' ' || pick({L(['Pro', 'Plus', 'Mini', 'XL', '2.0', 'Eco'])}, pid, 3) AS product_name,
        CASE WHEN rnd(pid, 41) < 0.03 THEN lower(cat) WHEN rnd(pid, 41) < 0.06 THEN upper(cat) WHEN rnd(pid, 41) < 0.08 THEN cat || ' ' ELSE cat END AS category,
        pick({L(SUBCATS)}, pid, 5) AS subcategory,
        pick({L(BRANDS)}, pid, 6) AS brand,
        1 + CAST(floor(rnd(pid, 9) * {N['suppliers']}) AS BIGINT) AS supplier_id,
        CAST(CASE WHEN rnd(pid * 3 + ver, 42) < 0.002 THEN -price ELSE price END AS DECIMAL(10, 2)) AS list_price,
        CAST(round(price * (0.55 + 0.25 * rnd(pid, 10)), 2) AS DECIMAL(10, 2)) AS cost_price,
        'EUR' AS currency,
        CASE WHEN rnd(pid, 11) < 0.05 THEN NULL ELSE '40' || lpad(CAST(pid AS VARCHAR), 11, '0') END AS ean,
        rnd(pid, 12) < 0.95 AS is_active,
        CASE WHEN op = 'I' AND {d} = 1 THEN TIMESTAMP '{ld}' + to_hours(6) ELSE TIMESTAMP '{ld}' + secs(rn, 13 + {d}) END AS updated_at,
        op
    FROM b
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("products", d, products_query(d))

## Tables 4 to 6: `suppliers`, `stores`, `promotions` (weekly snapshots)

These sources deliver a **full snapshot** every seven days (days 1, 8, 15, 22 and 29). Between snapshots nothing arrives. Over time some
suppliers drop out of the snapshot (the Silver layer has to mark them inactive), new ones appear and attributes change (payment terms, manager,
promotion windows).

In [ ]:
def suppliers_query(d):
    si = (d - 1) // 7
    S0, SN = N["suppliers"], N["suppliers_new"]
    ld = day_date(d)
    return f"""
    WITH ids AS (SELECT i AS sid FROM range(1, {S0} + {si} * {SN} + 1) t(i) WHERE {si} = 0 OR rnd(i, 5) >= 0.01 * {si})
    SELECT
        sid AS supplier_id,
        pick({L(SUP_A)}, sid, 1) || ' ' || pick({L(SUP_B)}, sid, 2) || ' ' || pick({L(LEGAL)}, sid, 3) AS supplier_name,
        pick({L(['DE'] * 8 + ['AT', 'CH', 'NL', 'PL', 'CZ', 'IT', 'Germany', 'de'])}, sid, 4) AS country,
        CASE WHEN rnd(sid, 6) < 0.05 THEN NULL WHEN rnd(sid, 6) < 0.08 THEN 'contact.at.supplier' || sid
             ELSE 'purchasing' || sid || '@' || lower(pick({L(SUP_A)}, sid, 1)) || '-' || lower(pick({L(SUP_B)}, sid, 2)) || '.example' END AS contact_email,
        pick({L([14, 30, 45, 60])}, sid, 20 + {si}) AS payment_terms_days,
        pick({L([1, 2, 3, 4, 5, 5, 4, 3, None])}, sid, 21 + {si}) AS rating,
        pick({L(['Y'] * 18 + ['N', 'y'])}, sid, 22 + {si}) AS active_flag,
        TIMESTAMP '{ld}' AS updated_at
    FROM ids
    """

In [ ]:
def stores_query(d):
    si = (d - 1) // 7
    ld = day_date(d)
    return f"""
    WITH ids AS (SELECT i AS sid FROM range(1, {N['stores']} + 1) t(i))
    SELECT
        sid AS store_id,
        CASE WHEN sid = 1 THEN 'Oog Online Shop' ELSE 'Oog ' || pick({L(CITIES)}, sid, 1) || ' ' || sid END AS store_name,
        CASE WHEN sid = 1 THEN 'online' ELSE pick({L(STORE_TYPES)}, sid, 2) END AS store_type,
        pick({L(CITIES)}, sid, 1) AS city,
        pick({L(['DE'] * 12 + ['AT', 'CH', 'NL'])}, sid, 3) AS country,
        pick({L(['north', 'south', 'east', 'west', 'central'])}, sid, 4) AS region,
        DATE '2008-01-01' + CAST(floor(rnd(sid, 5) * 6000) AS INTEGER) AS opened_on,
        pick({L(FIRST)}, sid, 6 + {si}) || ' ' || pick({L(LAST)}, sid, 7 + {si}) AS manager_name,
        CASE WHEN rnd(sid, 8) < 0.02 THEN NULL WHEN rnd(sid, 8) < 0.03 THEN 0 ELSE CAST(80 + floor(rnd(sid, 9) * 1900) AS INTEGER) END AS sqm,
        CASE WHEN sid = 1 THEN true ELSE rnd(sid, 10 + {si}) >= 0.03 * ({si} + 1) END AS is_open,
        TIMESTAMP '{ld}' AS updated_at
    FROM ids
    """

In [ ]:
def promotions_query(d):
    si = (d - 1) // 7
    ld = day_date(d)
    return f"""
    WITH ids AS (SELECT i AS pid FROM range(1, {N['promotions']} + {si} * {N['promotions_new']} + 1) t(i))
    SELECT
        pid AS promotion_id,
        CASE WHEN rnd(pid, 1) < 0.02 THEN lower('promo' || pid) ELSE 'PROMO' || pid END AS promo_code,
        pick({L(['Spring sale', 'Summer deal', 'Black week', 'Newsletter bonus', 'Free shipping', 'Clearance', 'Member day'])}, pid, 2) AS description,
        pick({L([5, 10, 10, 15, 20, 25, 30, 120])}, pid, 3) AS discount_pct,
        pick({L(['percent'] * 6 + ['PERCENT'])}, pid, 4) AS discount_type,
        DATE '2026-08-01' + CAST(floor(rnd(pid, 5) * 60) AS INTEGER) AS valid_from,
        CASE WHEN rnd(pid, 7) < 0.01 THEN DATE '2026-07-01'
             ELSE DATE '2026-08-01' + CAST(floor(rnd(pid, 5) * 60) AS INTEGER) + CAST(7 + floor(rnd(pid, 6) * 24) AS INTEGER) END AS valid_to,
        pick({L(['all', 'web', 'store', 'app'])}, pid, 8) AS channel,
        TIMESTAMP '{ld}' + to_hours({si}) AS updated_at
    FROM ids
    """

In [ ]:
for d in range(1, DAYS + 1):
    if is_snapshot_day(d):
        write_day("suppliers", d, suppliers_query(d))
        write_day("stores", d, stores_query(d))
        write_day("promotions", d, promotions_query(d))

## Helper: item rows and order totals

An order header carries its total and a payment carries its amount. To make orders, order items and payments fit together, the **order total is derived from the item rows**:
`item_select` is the single place that defines what an item looks like, and `ensure_totals` sums the items of a day per order and adds shipping. A small share of the totals
is wrong on purpose (the shop system adds a surcharge or rounds differently), and later item corrections and deletions make header and items drift apart, as they do in real systems.

In [ ]:
def new_item_ids(dd):
    ITM = N["items"]
    return f"SELECT i AS rn, {(dd - 1) * ITM} + i AS iid, 'I' AS op, {dd} AS dd, 0 AS ver FROM range(1, {ITM} + 1) t(i)"


def item_select(ids, d):
    """Item rows for an ids query with the columns rn, iid, op, dd, ver; d is the delivery day."""
    ORD = N["orders"]
    ld = day_date(d)
    return f"""
    WITH ids AS ({ids}),
    b AS (SELECT *, (dd - 1) * {ORD} + 1 + CAST(hash(iid) % {ORD} AS BIGINT) AS oid,
                 CASE WHEN rnd(iid, 2) < 0.0005 THEN 80000000 + iid % 100
                      ELSE 1 + CAST(floor(power(rnd(iid, 3), 2) * ({N['products']} + GREATEST(0, dd - 1) * {N['products_new']})) AS BIGINT) END AS pid
          FROM ids)
    SELECT
        iid AS item_id,
        oid AS order_id,
        pid AS product_id,
        CASE WHEN rnd(iid, 5) < 0.001 THEN 0 WHEN rnd(iid, 5) < 0.0015 THEN -1
             ELSE CAST(1 + floor(power(rnd(iid, 4 + ver), 3) * 5) AS INTEGER) END AS quantity,
        CASE WHEN rnd(iid, 16) < 0.005 THEN NULL WHEN rnd(iid, 16) < 0.006 THEN CAST(-base_price(pid) AS DECIMAL(10, 2))
             ELSE CAST(base_price(pid) AS DECIMAL(10, 2)) END AS unit_price,
        pick({L([0, 0, 0, 0, 5, 10, 15, 20])}, iid, 6 + ver) AS discount_pct,
        pick({L([19, 19, 19, 7])}, iid, 7) AS tax_rate,
        CASE WHEN op = 'I' THEN CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + secs(oid, 6)
             ELSE TIMESTAMP '{ld}' + secs(rn, 17 + {d}) END AS updated_at,
        op
    FROM b
    """


con.execute("CREATE OR REPLACE TABLE order_header (day INTEGER, order_id BIGINT, total_reported DECIMAL(12, 2), shipping DECIMAL(6, 2))")
_totals_ready = set()


def ensure_totals(first_day, last_day):
    """Header totals of the orders of the days first_day..last_day (computed once per day, kept in a DuckDB table)."""
    ORD = N["orders"]
    for dd in range(max(1, first_day), last_day + 1):
        if dd in _totals_ready:
            continue
        ship = f"pick({L([0, 0, 3.9, 4.9, 5.9])}, oid, 11)"
        con.execute(f"""
            INSERT INTO order_header
            WITH it AS (SELECT order_id, sum(quantity * unit_price * (1 - discount_pct / 100.0)) AS items_total
                        FROM ({item_select(new_item_ids(dd), dd)}) GROUP BY order_id),
                 o AS (SELECT {(dd - 1) * ORD} + i AS oid FROM range(1, {ORD} + 1) t(i))
            SELECT {dd}, oid,
                   CAST(round((COALESCE(it.items_total, 0) + {ship}) * CASE WHEN rnd(oid, 50) < 0.02 THEN 1.05 + rnd(oid, 51) * 0.1 ELSE 1 END, 2) AS DECIMAL(12, 2)),
                   CAST({ship} AS DECIMAL(6, 2))
            FROM o LEFT JOIN it ON it.order_id = o.oid""")
        _totals_ready.add(dd)

## Table 7: `orders` (daily transactions)

Every day: the new orders of that day (`I`), status changes of orders from the previous three days (`U`: paid, shipped, delivered, cancelled)
and a few deleted orders (`D`). Dirty: status in different spellings, some orders without a customer (guest checkout), a few customer ids that
do not exist, missing channel, lower case currency, duplicate rows. The header total (`order_total_reported`) is the sum of the order items plus shipping
as the shop system saw it on the order day; it differs for about 2 % of the orders and wherever items were corrected, deleted or are invalid later.

In [ ]:
def orders_query(d):
    ORD, NST, NPR = N["orders"], N["stores"], N["promotions"]
    new = f"SELECT i AS rn, {(d - 1) * ORD} + i AS oid, 'I' AS op, {d} AS dd, 0 AS age FROM range(1, {ORD} + 1) t(i)"
    parts = [new, f"SELECT rn, oid, op, dd, age FROM ({new}) WHERE rn <= {max(1, int(ORD * 0.002))}"]
    if d >= 2:
        span = min(3, d - 1)

        def past(count, op, salt):
            return (f"SELECT i AS rn, (dd - 1) * {ORD} + 1 + CAST(floor(rnd(i, {salt} + {d}) * {ORD}) AS BIGINT) AS oid, '{op}' AS op, dd, {d} - dd AS age "
                    f"FROM (SELECT i, {d} - 1 - CAST(floor(rnd(i, {salt} + 500 + {d}) * {span}) AS INTEGER) AS dd FROM range(1, {count} + 1) t(i))")

        parts.append(past(max(1, int(ORD * 0.03)), "U", 8100))
        parts.append(past(max(1, int(ORD * 0.001)), "D", 8300))
    ensure_totals(d - 3, d)
    ld = day_date(d)
    return f"""
    WITH ids AS ({" UNION ALL ".join(parts)}),
    b AS (SELECT ids.*, CASE WHEN rnd(oid, 3) < 0.01 THEN NULL ELSE pick({L(CHANNELS)}, oid, 3) END AS ch,
                 CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + secs(oid, 6) AS ots,
                 {N['customers']} + GREATEST(0, dd - 2) * {N['customers_new']} AS cust_n,
                 oh.total_reported, oh.shipping
          FROM ids LEFT JOIN order_header oh ON oh.day = ids.dd AND oh.order_id = ids.oid)
    SELECT
        oid AS order_id,
        CASE WHEN rnd(oid, 1) < 0.01 THEN NULL
             WHEN rnd(oid, 1) < 0.011 THEN 90000000 + oid % 1000
             ELSE 1 + CAST(floor(rnd(oid, 2) * cust_n) AS BIGINT) END AS customer_id,
        CASE WHEN ch IN ('web', 'app') THEN 1 ELSE 2 + CAST(floor(rnd(oid, 4) * {max(1, NST - 1)}) AS BIGINT) END AS store_id,
        ots AS order_ts,
        CASE op WHEN 'D' THEN 'cancelled'
                WHEN 'I' THEN CASE WHEN rnd(oid, 7) < 0.03 THEN 'PAID' WHEN rnd(oid, 7) < 0.05 THEN ' Paid'
                                   ELSE pick({L(['created'] * 4 + ['paid'])}, oid, 7) END
                ELSE CASE age WHEN 1 THEN pick({L(['paid', 'shipped'])}, oid, 9 + {d})
                              WHEN 2 THEN pick({L(['shipped', 'delivered'])}, oid, 9 + {d})
                              ELSE pick({L(['delivered', 'delivered', 'delivered', 'cancelled'])}, oid, 9 + {d}) END END AS status,
        ch AS channel,
        CASE WHEN rnd(oid, 15) < 0.02 THEN 'eur' ELSE 'EUR' END AS currency,
        total_reported AS order_total_reported,
        shipping AS shipping_cost,
        CASE WHEN rnd(oid, 12) < 0.2 THEN 1 + CAST(floor(rnd(oid, 13) * {NPR}) AS BIGINT) END AS promotion_id,
        CASE WHEN op = 'I' THEN ots ELSE TIMESTAMP '{ld}' + secs(rn, 14 + {d}) END AS updated_at,
        op
    FROM b
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("orders", d, orders_query(d))

## Table 8: `order_items` (daily transactions, the biggest table)

About one million new order lines per day. The mapping to orders is a pure function of the item id, so corrections of older items (quantity,
discount: `U`) and deletions (`D`) can reference them without any lookup. Dirty: quantity zero or negative, missing or negative prices,
product ids that do not exist, duplicate rows.

In [ ]:
def items_query(d):
    ITM = N["items"]
    new = new_item_ids(d)
    parts = [new, f"SELECT rn, iid, op, dd, ver FROM ({new}) WHERE rn <= {max(1, int(ITM * 0.001))}"]
    if d >= 2:
        span = min(3, d - 1)

        def past(count, op, salt):
            return (f"SELECT i AS rn, (dd - 1) * {ITM} + 1 + CAST(floor(rnd(i, {salt} + {d}) * {ITM}) AS BIGINT) AS iid, '{op}' AS op, dd, {d} AS ver "
                    f"FROM (SELECT i, {d} - 1 - CAST(floor(rnd(i, {salt} + 500 + {d}) * {span}) AS INTEGER) AS dd FROM range(1, {count} + 1) t(i))")

        parts.append(past(max(1, int(ITM * 0.001)), "U", 8500))
        parts.append(past(max(1, int(ITM * 0.0005)), "D", 8700))
    return item_select(" UNION ALL ".join(parts), d)

In [ ]:
for d in range(1, DAYS + 1):
    write_day("order_items", d, items_query(d))

## Table 9: `payments` (daily transactions)

One payment per paid order (about 3 % of the orders have none, e.g. cash on delivery). The payment id equals the order id. Later days bring
status changes (pending to captured, captured to refunded; `U`). Dirty: eleven spellings of six payment methods, about 1 % of the amounts differ
from the order total.

In [ ]:
def payments_query(d):
    ORD = N["orders"]
    new = (f"SELECT i AS rn, {(d - 1) * ORD} + i AS oid, 'I' AS op, {d} AS dd, 0 AS age FROM range(1, {ORD} + 1) t(i) "
           f"WHERE rnd({(d - 1) * ORD} + i, 20) >= 0.03")
    parts = [new]
    if d >= 2:
        span = min(3, d - 1)
        count = max(1, int(ORD * 0.02))
        parts.append(f"SELECT i AS rn, (dd - 1) * {ORD} + 1 + CAST(floor(rnd(i, 8900 + {d}) * {ORD}) AS BIGINT) AS oid, 'U' AS op, dd, {d} - dd AS age "
                     f"FROM (SELECT i, {d} - 1 - CAST(floor(rnd(i, 9400 + {d}) * {span}) AS INTEGER) AS dd FROM range(1, {count} + 1) t(i)) "
                     f"WHERE rnd((dd - 1) * {ORD} + 1 + CAST(floor(rnd(i, 8900 + {d}) * {ORD}) AS BIGINT), 20) >= 0.03")
    ensure_totals(d - 3, d)
    ld = day_date(d)
    return f"""
    WITH ids0 AS ({" UNION ALL ".join(parts)}),
    ids AS (SELECT ids0.*, oh.total_reported FROM ids0 LEFT JOIN order_header oh ON oh.day = ids0.dd AND oh.order_id = ids0.oid)
    SELECT
        oid AS payment_id,
        oid AS order_id,
        pick({L(PAY_RAW)}, oid, 21) AS payment_method,
        CAST(round(total_reported * CASE WHEN rnd(oid, 22) < 0.01 THEN 1.07 ELSE 1 END, 2) AS DECIMAL(12, 2)) AS amount,
        'EUR' AS currency,
        CASE op WHEN 'I' THEN pick({L(['captured'] * 6 + ['pending', 'failed'])}, oid, 23)
                ELSE CASE age WHEN 1 THEN 'captured' ELSE pick({L(['captured', 'refunded'])}, oid, 24 + {d}) END END AS status,
        least(CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + secs(oid, 6) + to_seconds(60 + CAST(floor(rnd(oid, 25) * 3000) AS BIGINT)),
              CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + to_seconds(86399)) AS paid_at,
        CASE WHEN op = 'I' THEN least(CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + secs(oid, 6) + to_seconds(60 + CAST(floor(rnd(oid, 25) * 3000) AS BIGINT)),
                                      CAST(DATE '{MONTH_START}' + CAST(dd - 1 AS INTEGER) AS TIMESTAMP) + to_seconds(86399))
             ELSE TIMESTAMP '{ld}' + secs(rn, 26 + {d}) END AS updated_at,
        op
    FROM ids
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("payments", d, payments_query(d))

## Table 10: `returns` (daily transactions)

About 10,000 return requests per day for order items of the last five days. Later days advance the status (requested, received, refunded; `U`).
Dirty: missing reason, quantity larger than the ordered amount is possible, refund amounts that do not fit the item price.

In [ ]:
def returns_query(d):
    ORD, ITM, RET = N["orders"], N["items"], N["returns"]
    new = (f"SELECT m AS rn, {(d - 1) * RET} + m AS rid, 'I' AS op, "
           f"GREATEST(1, {d} - CAST(floor(rnd(m, 3000 + {d}) * 5) AS INTEGER)) AS dd FROM range(1, {RET} + 1) t(m)")
    parts = [new]
    if d >= 2:
        span = min(5, d - 1)
        count = max(1, int(RET * 0.05))
        parts.append(f"SELECT i AS rn, (dd - 1) * {RET} + 1 + CAST(floor(rnd(i, 9100 + {d}) * {RET}) AS BIGINT) AS rid, 'U' AS op, dd "
                     f"FROM (SELECT i, {d} - 1 - CAST(floor(rnd(i, 9200 + {d}) * {span}) AS INTEGER) AS dd FROM range(1, {count} + 1) t(i))")
    ld = day_date(d)
    return f"""
    WITH ids AS ({" UNION ALL ".join(parts)}),
    b AS (SELECT *, (dd - 1) * {ITM} + 1 + CAST(floor(rnd(rid, 30) * {ITM}) AS BIGINT) AS iid FROM ids),
    c AS (SELECT *, (dd - 1) * {ORD} + 1 + CAST(hash(iid) % {ORD} AS BIGINT) AS oid,
                 1 + CAST(floor(power(rnd(iid, 3), 2) * ({N['products']} + GREATEST(0, dd - 1) * {N['products_new']})) AS BIGINT) AS pid,
                 1 + CAST(floor(rnd(rid, 31) * 2) AS INTEGER) AS qty FROM b)
    SELECT
        rid AS return_id,
        iid AS item_id,
        oid AS order_id,
        qty AS quantity,
        pick({L(RETURN_REASONS)}, rid, 32) AS reason,
        CAST(round(base_price(pid) * qty * CASE WHEN rnd(rid, 33) < 0.02 THEN 1.5 ELSE 1 END, 2) AS DECIMAL(10, 2)) AS refund_amount,
        CASE op WHEN 'I' THEN 'requested' ELSE pick({L(['received', 'refunded', 'refunded'])}, rid, 34 + {d}) END AS status,
        TIMESTAMP '{ld}' + secs(rid, 35 + {d}) AS return_ts,
        TIMESTAMP '{ld}' + secs(rid, 35 + {d}) AS updated_at,
        op
    FROM c
    """

In [ ]:
for d in range(1, DAYS + 1):
    write_day("returns", d, returns_query(d))

## Summary

What is in the Bronze folder now: number of files and rows per table, read from the Parquet metadata in S3.

In [ ]:
con.execute(f"""
    SELECT split_part(replace(file_name, '{BRONZE}/', ''), '/', 1) AS table_name,
           count(*) AS files, sum(num_rows) AS rows, min(num_rows) AS min_rows_per_file, max(num_rows) AS max_rows_per_file
    FROM parquet_file_metadata('{BRONZE}/*/*/*.parquet')
    GROUP BY 1 ORDER BY rows DESC
""").df()

Next: `02-medallion-pipeline.ipynb` loads one load date at a time (Bronze, Silver, Gold). `03-medallion-cleanup.ipynb` resets the schema
evolution or the whole demo again.